In [3]:
import pandas as pd
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def scrape_phong_vu(so_luong_can_lay):
    print(f"Bắt đầu quét dữ liệu -> Mục tiêu: {so_luong_can_lay} sản phẩm.")
    
    chrome_options = Options()
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-notifications")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    results = []
    page = 1

    try:
        while len(results) < so_luong_can_lay:
            url_listing = f"https://phongvu.vn/c/laptop?page={page}"
            print(f"--- Đang quét trang {page} ---")
            driver.get(url_listing)
            
            try:
                WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.product-card")))
            except:
                print("Không tìm thấy sản phẩm hoặc đã hết trang.")
                break

            for _ in range(4):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(1)

            cards = driver.find_elements(By.CSS_SELECTOR, "div.product-card")
            
            for card in cards:
                if len(results) >= so_luong_can_lay:
                    break
                
                try:
                    # 1. Lấy Tên và cắt bỏ mọi thứ từ dấu ngoặc đơn '(' trở đi
                    ten_raw = card.find_element(By.CSS_SELECTOR, "div.att-product-card-title h3").text.strip()
                    ten = ten_raw.split('(')[0].strip()
                    
                    # 2. Lấy Giá Đang Bán (Hiện tại)
                    try:
                        gia_ht_raw = card.find_element(By.CSS_SELECTOR, "div.att-product-detail-latest-price").text.strip()
                        gia_hien_tai = re.sub(r"[^\d]", "", gia_ht_raw)
                    except:
                        gia_hien_tai = "0"

                    # 3. Lấy Giá Gốc (Gạch ngang)
                    try:
                        gia_goc_raw = card.find_element(By.XPATH, ".//del | .//*[contains(@class, 'retail-price')]").text.strip()
                        gia_goc = re.sub(r"[^\d]", "", gia_goc_raw)
                    except:
                        gia_goc = gia_hien_tai # Nếu không có giảm giá thì Giá gốc = Giá hiện tại

                    # 4. Lấy URL
                    url_sp = card.find_element(By.XPATH, ".//a").get_attribute("href")

                    if ten and url_sp and gia_hien_tai != "0":
                        results.append({
                            "Tên sản phẩm": ten,
                            "Giá gốc": gia_goc,
                            "Giá hiện tại": gia_hien_tai,
                            "URL": url_sp
                        })
                        # Đã xóa phần [:40]... ở đây để in ra tên gọn gàng
                        print(f"[{len(results)}] Đã lấy: {ten} | Gốc: {gia_goc} -> Bán: {gia_hien_tai}")
                        
                except Exception as e:
                    continue
            
            page += 1
            time.sleep(1)

    finally:
        driver.quit()

    return pd.DataFrame(results)

if __name__ == "__main__":
    SO_LUONG = 10
    OUTPUT_FILE = "Laptop_PhongVu.csv"

    df = scrape_phong_vu(SO_LUONG)

    if not df.empty:
        df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
        print(f"\nTHÀNH CÔNG! Đã lưu {len(df)} sản phẩm vào file: {OUTPUT_FILE}")
    else:
        print("\nKhông lấy được dữ liệu.")

Bắt đầu quét dữ liệu -> Mục tiêu: 10 sản phẩm.
--- Đang quét trang 1 ---
[1] Đã lấy: Laptop Msi Katana B14WEK-286VN | Gốc: 41990000 -> Bán: 33190000
[2] Đã lấy: Laptop Asus Vivobook S14 S3407CA-LY068WS | Gốc: 27990000 -> Bán: 23490000
[3] Đã lấy: Laptop Lenovo IdeaPad Slim 3 15ARP10 - 83K700GDVN | Gốc: 19900000 -> Bán: 19900000
[4] Đã lấy: Laptop HP 15-fc0024AU - D0BH2PA | Gốc: 17590000 -> Bán: 16490000
[5] Đã lấy: Laptop HP Victus 15 fa2731TX | Gốc: 29990000 -> Bán: 27190000
[6] Đã lấy: Laptop HP OmniBook 7 14-fr0027TU - C1MN1PA | Gốc: 32290000 -> Bán: 32290000
[7] Đã lấy: Laptop Lenovo Legion 5 15IPH11 - 83RW0023VN | Gốc: 60990000 -> Bán: 60990000
[8] Đã lấy: Laptop HP OmniBook 5 16 ag1069AU - BZ7T1PA | Gốc: 27190000 -> Bán: 27190000
[9] Đã lấy: Laptop HP OmniBook 7 14-fs0042TU - C1MP3PA | Gốc: 30190000 -> Bán: 30190000
[10] Đã lấy: Laptop HP 15-fd1289TU - C2CV8PA | Gốc: 26590000 -> Bán: 26590000

THÀNH CÔNG! Đã lưu 10 sản phẩm vào file: Laptop_PhongVu.csv
